<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; EDM&plus; Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">EDM&plus; NB07 &mdash; Data Quality Workflow &amp; Derived Properties</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Proves derived properties resolve correctly and drives data-quality exceptions through the Workflow dashboard end to end.</div>
</div>

<sub>EDM&plus; pack sequence: NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; <b>NB07</b></sub>

# NB07: Data Quality Workflow & Derived Properties

**What this does:** 
1. Proves derived properties work by setting and removing an override on Apple
2. Creates a Workflow Task Definition for DQ review
3. Loads 5 new instruments with intentional data quality issues
4. Creates Workflow tasks for each DQ failure
5. You resolve them manually in the Workflow dashboard

**Business context:** In production, DQ validation runs automatically when files land in Drive. Failures create Workflow tasks for the operations team to review and resolve. This exercise walks you through the exact same process manually.

**Verify after running:** Go to Workflow Service → Dashboard. You should see 4 tasks in state "New".

In [ ]:
# Run this cell first — installs required packages (takes ~30 seconds, only needed once per session)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lusid-sdk', 'lusid-workflow-sdk', 'finbourne-sdk-utilities', 'lusid-drive-sdk', 'lumipy'])
print("✅ Packages installed")

In [ ]:
# ============================================================================
# CREDENTIALS: edit secrets.json (NOT this notebook)
# ============================================================================
# Copy secrets.json.example to secrets.json and fill in your domain + PAT:
#   {
#     "api_url": "https://<YOUR_DOMAIN>.lusid.com/api",
#     "personal_access_token": "<YOUR_PAT>"
#   }
# To get a PAT: LUSID web app → profile icon (top right) → Personal Access Tokens → Create
# (Override the file location with the EDM_SECRETS_PATH environment variable.)

import json as _json, os as _os
_secrets_path = _os.environ.get("EDM_SECRETS_PATH", "secrets.json")
with open(_secrets_path) as _f:
    _secrets = _json.load(_f)
api_url = _secrets["api_url"]
personal_access_token = _secrets["personal_access_token"]

# ============================================================================
# Imports — do not edit below this line
# ============================================================================
import os, json
from datetime import datetime, timedelta, date, timezone
import datetime as dt
import pandas as pd
import pytz

import lusid as lu
import lusid.models as lm

try:
    import lusid_workflow as lwf
    import lusid_workflow.models as wf_models
except ImportError:
    lwf = None
    wf_models = None
    print("⚠️ lusid_workflow not available")

import lumipy as lp

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.4f}".format

# ============================================================================
# Authentication
# ============================================================================
if "<YOUR_DOMAIN>" in api_url or "<YOUR_PAT>" in personal_access_token or not personal_access_token:
    raise ValueError(
        "\n\n⛔ You need to edit the two lines at the top of this cell:\n"
        "   1. Replace <YOUR_DOMAIN> with your LUSID domain name\n"
        "   2. Replace <YOUR_PAT> with your Personal Access Token\n\n"
        "   To get a PAT: LUSID web app → profile icon → Personal Access Tokens → Create")

def get_factory(app='lusid'):
    module = __import__(app)
    # Each service has a different URL path
    url_map = {'lusid': '/api', 'lusid_workflow': '/workflow', 'lusid_drive': '/drive'}
    service_url = api_url.replace('/api', url_map.get(app, '/api'))
    config_loaders = [module.extensions.ArgsConfigurationLoader(
        api_url=service_url, access_token=personal_access_token)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    luminesce_url = api_url.replace('/api', '/honeycomb')
    return lp.get_client(access_token=personal_access_token, api_url=luminesce_url)

# Initialise
lusid_factory = get_factory('lusid')

# Verify connection
try:
    api_instance = lusid_factory.build(lu.ApplicationMetadataApi)
    api_response = api_instance.get_lusid_versions()
    domain = api_response.links[0].href
    env_url = domain[0:domain.find('/app/')]
    print(f"✅ Connected to {env_url}")
    print(f"   API Version: {api_response.build_version}")
except Exception as e:
    print(f"⚠️ Connected but could not verify: {e}")

lumi = get_lumi()
print("✅ Luminesce ready")

---
## Part 1: Prove Derived Properties Work

In [ ]:
instruments_api = lusid_factory.build(lu.InstrumentsApi)
SCOPE = 'edm-training2'

print("--- Step 1: Read Apple's current credit ratings ---")
try:
    resp = instruments_api.get_instrument(
        identifier_type='ClientInternal', identifier='EQT00001', scope='default',
        property_keys=[
            f'Instrument/{SCOPE}/RatingS', f'Instrument/{SCOPE}/CreditRating',
            'Instrument/Master/MasterCreditRating',
        ])
    print(f"  Instrument: {resp.name}")
    for prop in (resp.properties or []):
        val = prop.value.label_value if prop.value and prop.value.label_value else (prop.value.metric_value if prop.value else None)
        print(f"    {prop.key}: {val}")
except lu.ApiException as e:
    print(f"  Error: {e.reason}")

print("\n--- Step 2: Set a credit rating override ---")
try:
    instruments_api.upsert_instruments_properties(
        upsert_instrument_property_request=[
            lm.UpsertInstrumentPropertyRequest(
                identifier_type='ClientInternal', identifier='EQT00001',
                properties=[lm.ModelProperty(
                    key='Instrument/Override/CreditRating_Override',
                    value=lm.PropertyValue(label_value='AA+'))])])
    print("  Set Override/CreditRating_Override = 'AA+' on Apple")
except lu.ApiException as e:
    print(f"  Error: {e.reason}")

print("\n--- Step 3: Read Apple again ---")
try:
    resp = instruments_api.get_instrument(
        identifier_type='ClientInternal', identifier='EQT00001', scope='default',
        property_keys=[
            f'Instrument/{SCOPE}/RatingS', f'Instrument/{SCOPE}/CreditRating',
            'Instrument/Override/CreditRating_Override',
            'Instrument/Master/MasterCreditRating',
        ])
    print(f"  Instrument: {resp.name}")
    for prop in (resp.properties or []):
        val = prop.value.label_value if prop.value and prop.value.label_value else (prop.value.metric_value if prop.value else None)
        marker = ' ◄ OVERRIDE' if 'Override' in prop.key else (' ◄ DERIVED' if 'Master' in prop.key else '')
        print(f"    {prop.key}: {val}{marker}")
except lu.ApiException as e:
    print(f"  Error: {e.reason}")

print("\n--- Step 4: Remove the override ---")
try:
    instruments_api.delete_instrument_properties(
        identifier_type='ClientInternal', identifier='EQT00001',
        request_body=['Instrument/Override/CreditRating_Override'])
    print("  Removed override. MasterCreditRating will revert to source value.")
except lu.ApiException as e:
    print(f"  Note: {e.reason}")

---
## Part 2: Create Workflow Task Definition

In [ ]:
try:
    workflow_factory = get_factory('lusid_workflow')
    task_defs_api = workflow_factory.build(lwf.TaskDefinitionsApi)
    tasks_api = workflow_factory.build(lwf.TasksApi)
    print("✅ Workflow Service connected")
except Exception as e:
    print(f"❌ Workflow Service not available: {e}")
    task_defs_api = None

TASK_DEF_SCOPE = SCOPE
TASK_DEF_CODE = 'DQ-Review'

if task_defs_api:
    print(f"\n--- Creating Task Definition: {TASK_DEF_SCOPE}/{TASK_DEF_CODE} ---")
    try:
        task_defs_api.delete_task_definition(scope=TASK_DEF_SCOPE, code=TASK_DEF_CODE)
        print("  Deleted existing definition")
    except: pass
    try:
        task_def = task_defs_api.create_task_definition(
            create_task_definition_request=wf_models.CreateTaskDefinitionRequest(
                id=wf_models.ResourceId(scope=TASK_DEF_SCOPE, code=TASK_DEF_CODE),
                display_name='DQ Review',
                description='Review and resolve data quality issues on new instruments',
                states=[
                    wf_models.TaskStateDefinition(name='New', display_name='New'),
                    wf_models.TaskStateDefinition(name='InProgress', display_name='In Progress'),
                    wf_models.TaskStateDefinition(name='Resolved', display_name='Resolved'),
                    wf_models.TaskStateDefinition(name='Rejected', display_name='Rejected'),
                ],
                initial_state=wf_models.InitialState(name='New', required_fields=[]),
                field_schema=[
                    wf_models.TaskFieldDefinition(name='InstrumentId', type='String', display_name='Instrument ID'),
                    wf_models.TaskFieldDefinition(name='InstrumentName', type='String', display_name='Instrument Name'),
                    wf_models.TaskFieldDefinition(name='DQIssue', type='String', display_name='DQ Issue'),
                    wf_models.TaskFieldDefinition(name='DQRule', type='String', display_name='DQ Rule'),
                    wf_models.TaskFieldDefinition(name='Resolution', type='String', display_name='Resolution'),
                ],
                triggers=[
                    wf_models.TransitionTriggerDefinition(name='start-review', trigger=wf_models.TriggerSchema(type='External'), display_name='Start Review'),
                    wf_models.TransitionTriggerDefinition(name='resolve', trigger=wf_models.TriggerSchema(type='External'), display_name='Mark Resolved'),
                    wf_models.TransitionTriggerDefinition(name='reject', trigger=wf_models.TriggerSchema(type='External'), display_name='Reject'),
                    wf_models.TransitionTriggerDefinition(name='reject-new', trigger=wf_models.TriggerSchema(type='External'), display_name='Reject Without Review'),
                ],
                transitions=[
                    wf_models.TaskTransitionDefinition(from_state='New', to_state='InProgress', trigger='start-review', display_name='Start Review'),
                    wf_models.TaskTransitionDefinition(from_state='InProgress', to_state='Resolved', trigger='resolve', display_name='Mark Resolved'),
                    wf_models.TaskTransitionDefinition(from_state='InProgress', to_state='Rejected', trigger='reject', display_name='Reject'),
                    wf_models.TaskTransitionDefinition(from_state='New', to_state='Rejected', trigger='reject-new', display_name='Reject Without Review'),
                ],
            ))
        print(f"  ✅ Created: {task_def.id.scope}/{task_def.id.code}")
    except Exception as e:
        body = e.body[:400] if hasattr(e, 'body') and e.body else str(e)
        print(f"  ❌ Error: {body}")

---
## Part 3: Load Test Instruments with DQ Validation

In [ ]:
DATA_DIR = 'data/'
df = pd.read_csv(f'{DATA_DIR}new-instruments-dq-test.csv')
print(f"Loaded {len(df)} test instruments\n")

VALID_CURRENCIES = ['USD', 'EUR', 'GBP', 'CHF', 'JPY', 'AUD', 'CAD', 'ZAR', 'NZD', 'SEK', 'NOK', 'DKK']
VALID_RATINGS = ['AAA','AA+','AA','AA-','A+','A','A-','BBB+','BBB','BBB-','BB+','BB','BB-',
                 'B+','B','B-','CCC+','CCC','CCC-','CC','C','D','NR','WD']
NUMERIC_PROPERTIES = {'EsgEnvironmentRating', 'EsgSocialRating', 'EsgGovernanceRating'}

dq_results = []
for idx, row in df.iterrows():
    ci = str(row['ClientInternal']).strip()
    name = str(row['CompanyName']).strip()
    issues = []
    ccy = str(row.get('Currency', '')).strip()
    if ccy not in VALID_CURRENCIES:
        issues.append(('Invalid currency', f'Currency "{ccy}" is not a valid ISO code'))
    rating = str(row.get('CreditRating', '')).strip()
    if not rating or rating == 'nan':
        issues.append(('Missing credit rating', 'CreditRating is required but empty'))
    for esg_col in ['EsgEnvironmentRating', 'EsgSocialRating', 'EsgGovernanceRating']:
        val = row.get(esg_col)
        if pd.notna(val) and (float(val) < 0 or float(val) > 100):
            issues.append((f'{esg_col} out of range', f'{esg_col} = {val} (must be 0 to 100)'))
    status = 'PASS' if not issues else 'FAIL'
    icon = '✅' if status == 'PASS' else '❌'
    print(f"  {icon} {ci} ({name}): {status}")
    for rule, desc in issues:
        print(f"      Issue: {desc}")
    dq_results.append({'ci': ci, 'name': name, 'status': status, 'issues': issues})

passed = [r for r in dq_results if r['status'] == 'PASS']
failed = [r for r in dq_results if r['status'] == 'FAIL']
print(f"\nPassed: {len(passed)}, Failed: {len(failed)}")

---
## Part 4: Load Instruments and Create Workflow Tasks

In [ ]:
prop_map = {
    'Sector': 'Sector', 'Exchange': 'Exchange', 'ListingCountry': 'ListingCountry',
    'CountryIso': 'CountryIso', 'CreditRating': 'CreditRating',
    'EsgEnvironmentRating': 'EsgEnvironmentRating', 'EsgSocialRating': 'EsgSocialRating',
    'EsgGovernanceRating': 'EsgGovernanceRating', 'DataOwner': 'DataOwner',
    'DataGovernanceCode': 'DataGovernanceCode',
}

def build_props_dq(row):
    props = {}
    for csv_col, prop_code in prop_map.items():
        val = row.get(csv_col)
        if pd.isna(val) or str(val).strip() == '': continue
        key = f'Instrument/{SCOPE}/{prop_code}'
        if prop_code in NUMERIC_PROPERTIES:
            try:
                props[key] = lm.ModelProperty(key=key, value=lm.PropertyValue(
                    metric_value=lm.MetricValue(value=float(val), unit='')))
            except: props[key] = lm.ModelProperty(key=key, value=lm.PropertyValue(label_value=str(val).strip()))
        else:
            clean_val = str(val).strip()
            if clean_val.endswith('.0') and clean_val[:-2].isdigit():
                clean_val = clean_val[:-2]
            props[key] = lm.ModelProperty(key=key, value=lm.PropertyValue(label_value=clean_val))
    return props

print("--- Loading instruments ---")
all_defs = {}
for r in dq_results:
    ci = r['ci']
    row = df[df['ClientInternal'] == ci].iloc[0]
    ccy = str(row['Currency']).strip()
    if ccy not in VALID_CURRENCIES: ccy = 'USD'
    properties = build_props_dq(row)
    dq_status = 'Completed' if r['status'] == 'PASS' else 'PendingUpdate'
    properties[f'Instrument/{SCOPE}/DataQualityStatus'] = lm.ModelProperty(
        key=f'Instrument/{SCOPE}/DataQualityStatus', value=lm.PropertyValue(label_value=dq_status))
    all_defs[ci] = lm.InstrumentDefinition(
        name=str(row['CompanyName']).strip(),
        identifiers={'ClientInternal': lm.InstrumentIdValue(value=ci)},
        definition=lm.Equity(instrument_type='Equity', dom_ccy=ccy),
        properties=list(properties.values()))

try:
    resp = instruments_api.upsert_instruments(request_body=all_defs, scope='default')
    for key in resp.values:
        dq = 'Completed' if any(r['ci'] == key and r['status'] == 'PASS' for r in dq_results) else 'PendingUpdate'
        icon = '✅' if dq == 'Completed' else '⚠️'
        print(f"  {icon} {key}: loaded (DQ: {dq})")
    if resp.failed:
        for key, err in resp.failed.items():
            print(f"  ❌ {key}: {str(err)[:200]}")
except lu.ApiException as e:
    print(f"  ERROR: {e.reason}")

# Create Workflow tasks
if task_defs_api:
    print("\n--- Creating Workflow Tasks ---")
    tasks_created = 0
    for r in failed:
        for rule_name, issue_desc in r['issues']:
            try:
                task = tasks_api.create_task(
                    create_task_request=wf_models.CreateTaskRequest(
                        task_definition_id=wf_models.ResourceId(scope=TASK_DEF_SCOPE, code=TASK_DEF_CODE),
                        fields=[
                            wf_models.TaskInstanceField(name='InstrumentId', value=r['ci']),
                            wf_models.TaskInstanceField(name='InstrumentName', value=r['name']),
                            wf_models.TaskInstanceField(name='DQIssue', value=issue_desc),
                            wf_models.TaskInstanceField(name='DQRule', value=rule_name),
                        ],
                        correlation_ids=[f'dq-{r["ci"]}-{rule_name}']))
                tasks_created += 1
                print(f"  ✅ Task: {r['ci']} — {rule_name}")
            except Exception as e:
                body = e.body[:200] if hasattr(e, 'body') and e.body else str(e)
                print(f"  ❌ {r['ci']}: {body}")
    print(f"\n  {tasks_created} workflow tasks created")

print(f"\n✅ NB07 Complete")
print("\nNow go to Workflow Service → Dashboard to resolve the tasks.")
print("See the HTML training guide for step by step resolution instructions.")